<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">La Monitorización de la Pérdida y las Métricas</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">Entrenar bien exige observar el proceso mientras ocurre. Este notebook se centra en la monitorización de pérdida y métricas como forma de leer la evolución real del aprendizaje.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">Sin seguimiento, es difícil distinguir mejora real, estancamiento, inestabilidad o sobreajuste.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A interpretar señales de entrenamiento y validación para tomar decisiones más informadas sobre el proceso.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">Mirar curvas no es decorar resultados; es analizar el comportamiento dinámico del modelo.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">Pérdida y métricas forman series temporales que ayudan a diagnosticar cómo evoluciona la red con cada época o iteración.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Tensores de entrada y etiquetas
"""Este bloque crea tensores que representan las entradas y las etiquetas del conjunto de entrenamiento.
Este bloque importa "torch" para trabajar con tensores en PyTorch.
La variable "entradas" contiene muestras bidimensionales en forma de tensor que sirven como datos de entrada.

Aplicamos el método ".unsqueeze()" a las etiquetas para que tengan una dimensión adicional, convirtiéndolas en un tensor columna.
El argumento "1" indica que se debe agregar una dimensión adicional en la posición 1 del tensor.
La variable "etiquetas" contiene valores asociados a cada muestra en forma de tensor con una dimensión adicional en 
la posición 1 y define el objetivo del entrenamiento.
Estos tensores permiten observar cómo la pérdida evoluciona durante el entrenamiento."""

import torch

entradas = torch.tensor([[1.0, 2.0],
                         [3.0, 4.0],
                         [5.0, 6.0]])

etiquetas = torch.tensor([0.0, 1.0, 0.0]).unsqueeze(1)

entradas, etiquetas # Muestra los tensores de entrada y las etiquetas para verificar su forma y contenido.


In [ ]:
# Bloque 2 — Definición de la red neuronal
"""Este bloque define una red neuronal artificial utilizando nn.Module.
Este bloque importa "torch.nn" para definir módulos y capas de redes neuronales.

Definimos una clase personalizada llamada "RedMonitorizacion()" que representa un modelo de red neuronal simple.
La clase "RedMonitorizacion()" hereda de "nn.Module", que es la clase base utilizada para construir redes neuronales en PyTorch.

Definimos la función "def __init__()", que inicializa la estructura interna del modelo.
El parámetro "self" de la función hace referencia a la instancia actual de la clase "RedMonitorizacion()".

Utilizamos "super().__init__()" para invocar el constructor de la clase base "nn.Module".     
Esto asegura que la clase base se inicialice correctamente cada vez que se crea una instancia del modelo.

Aplicamos la clase  "nn.Linear()" a la variable de instancia "self.capa1" para definir la primera capa lineal.
Los argumentos "2, 4" de "nn.Linear()" indican que la capa lineal recibe 2 entradas y produce 4 salidas.
Aplicamos la clase  "nn.Linear()" a la variable de instancia "self.capa2" para definir la segunda capa lineal.
Los argumentos "4, 1" de "nn.Linear()" indican que la capa lineal recibe 4 entradas y produce 1 salida.

Definimos la función "def forward()" que especifica cómo fluye la entrada a través de las capas del modelo.
El método ".forward()" especifica cómo se propaga la información a través de la red.
Los parámetros "self" y "x" representan la instancia del modelo y los datos de entrada, respectivamente.

Asignamos la llamada a la función "self.capa1()" a la variable "x", aplicando la primera capa lineal a la entrada.
Aplicamos la función de activación "torch.relu()" a la variable "x" para introducir no linealidad en la red.
Asignamos la llamada a la función "self.capa2()" a la variable "x", aplicando la segunda capa lineal a la salida de la activación.
Utilizamos la instrucción "return" para devolver la salida del modelo tras aplicar las capas y la activación a la entrada.

En todos los casos se pasa "x" como parámetro a las funciones correspondientes para que la información fluya a través de la red.
Además, la variable "x" se actualiza con la salida de cada operación, asegurando que la información fluya correctamente a través de la red.
La variable "red" almacena una instancia de la clase "RedMonitorizacion()", representando el modelo de red neuronal definido.
La estructura permite generar predicciones que serán evaluadas mediante pérdida y métricas."""

import torch.nn as nn

class RedMonitorizacion(nn.Module):
    def __init__(self):
        super().__init__()
        self.capa1 = nn.Linear(2, 4)
        self.capa2 = nn.Linear(4, 1)

    def forward(self, x):
        x = self.capa1(x)
        x = torch.relu(x)
        x = self.capa2(x)
        return x

red = RedMonitorizacion()

red # Muestra la estructura del modelo de red neuronal para verificar su definición.


In [ ]:
# Bloque 3 — Función de pérdida y métrica
"""Este bloque define la función de pérdida y la métrica utilizadas durante el entrenamiento.
Aplicamos la clase "nn.BCEWithLogitsLoss()" a la variable "criterio" para crear un objeto calculador de función de pérdida.
La variable "criterio" almacena el objeto calculador de la función de pérdida, que recibe los logits de la red y las etiquetas.

Definimos la función "def exactitud()" para calcular la exactitud de las predicciones del modelo en comparación con las etiquetas reales.
Los parámetros "pred" y "etiquetas" representan los logits del modelo y las etiquetas reales, respectivamente.

Asignamos la expresión "(torch.sigmoid(pred) > 0.5).float()" a la variable "pred_binarias".
La expresión se compone de dos partes: primero aplicamos la función "torch.sigmoid()" a la variable "pred" para obtener probabilidades y luego 
comparamos esas probabilidades con "0.5" para obtener predicciones binarias, convirtiendo el resultado final al tipo "float".

Asignamos la expresión "(pred_binarias == etiquetas).float().mean()" a la variable "coincidencias".
La expresión se compone de dos partes: primero comparamos "pred_binarias" con "etiquetas" para obtener un tensor de coincidencias
booleanas; después convertimos ese tensor al tipo "float" y calculamos la media con el método "mean()" para obtener la proporción de aciertos.

Además, utilizamos la instrucción "return" para devolver la exactitud como un valor numérico de tipo "float", mediante la expresión 
"float(coincidencias.item())".
La métrica de exactitud calcula la proporción de predicciones correctas tras aplicar una función sigmoide."""

criterio = nn.BCEWithLogitsLoss()

def exactitud(pred, etiquetas):
    pred_binarias = (torch.sigmoid(pred) > 0.5).float()
    coincidencias = (pred_binarias == etiquetas).float().mean()
    return float(coincidencias.item())

criterio # Muestra la función de pérdida definida.


In [ ]:
# Bloque 4 — Definición del optimizador
"""Este bloque define el optimizador utilizado durante el entrenamiento.
Aplicamos la clase "torch.optim.Adam()" a la variable "optimizador" para crear un objeto optimizador de parámetros de la red.
Pasamos como argumentos los parámetros de la red mediante la expresión "red.parameters()".
El método ".parameters()" devuelve un iterador sobre los parámetros de la red que serán actualizados por el optimizador.
El parámetro "lr=" especifica la tasa de aprendizaje utilizada por el optimizador; en este caso, 0.01.
La variable "optimizador" almacena el objeto que gestiona las actualizaciones del modelo."""

optimizador = torch.optim.Adam(red.parameters(), lr=0.01)

optimizador # Muestra el optimizador definido.


In [ ]:
# Bloque 5 — Loop de entrenamiento con monitorización
"""Este bloque implementa el loop de entrenamiento con monitorización de la pérdida y de las métricas.
La variable "epocas" define cuántas veces se repite el ciclo completo; en este caso, 15 veces.
La variable "historial_perdida" se inicializa como una lista vacía "[]" para almacenar los valores de pérdida en cada época.
La variable "historial_exactitud" se inicializa como una lista vacía "[]" para almacenar los valores de exactitud en cada época.

Utilizamos un bucle "for" para iterar sobre el número de épocas definido en la variable "epocas"; es decir, se repetirá 15 veces en este caso.
La expresión "for _ in range(epocas):" inicia el bucle que se repetirá el número de veces definido en "epocas".

En cada iteración se ejecuta el forward pass, se calcula la pérdida y la exactitud, se reinician los gradientes previos,
se ejecuta backward para obtener los gradientes y se actualizan los parámetros con el optimizador.
Además, en cada iteración se almacenan la pérdida y la exactitud en las listas "historial_perdida" e "historial_exactitud", respectivamente.
Los valores registrados corresponden a las predicciones calculadas antes de actualizar los parámetros en cada época.
La métrica permite evaluar el rendimiento sobre los datos de entrenamiento de forma continua."""

epocas = 15
historial_perdida = []
historial_exactitud = []

for _ in range(epocas):
    pred = red(entradas)
    perdida = criterio(pred, etiquetas)
    acc = exactitud(pred, etiquetas)

    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()

    historial_perdida.append(float(perdida.item()))
    historial_exactitud.append(acc)


In [ ]:
# Bloque 6 — Resultados finales
"""Este bloque organiza los resultados del proceso de monitorización.
La variable "resultado" almacena en forma de diccionario el historial de pérdidas y métricas por época.
La estructura permite visualizar cómo evoluciona el rendimiento del modelo durante el entrenamiento."""

resultado = {
    "perdidas_por_epoca": historial_perdida,
    "exactitud_por_epoca": historial_exactitud
}

resultado # Muestra el diccionario con los resultados finales del entrenamiento.


<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto que entrenar también implica observar y diagnosticar el proceso con criterio.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">Las curvas de pérdida y métricas ayudan a detectar problemas que no se ven mirando una única cifra final.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Quedarse con el resultado final sin analizar la trayectoria que siguió el modelo para llegar hasta él.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en el Data Augmentation para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en el Data Augmentation y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">08_Data_augmentation/01_El_data_augmentation.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">